Полезные ссылки
Урок в прозе: https://proproprogs.ru/python_oop/python-oop-data-classes-pri-nasledovanii

Телеграм-канал: https://t.me/python_selfedu

In [2]:

from dataclasses import dataclass, field, InitVar
from typing import Any

@dataclass
class Goods:
    uid: Any
    price: Any = None
    weight: Any = None

# это эквивалентно следующему: def __init__(self, uid: Any, price: Any = None, weight: Any = None)

@dataclass
class Book(Goods):
    title: str = ''
    author: str = ''
    price: float = 0
    weight: int|float = 0

# это эквивалентно следующему: def __init__(self, uid: Any, price: float = 0, weight: int|float = 0, title: str = '', author: str = '')
# uid, price, weight остаются, но у price, weight переопределяется тип и значение по умолчанию. В конце добавляются новые атрибуты, присущие только классу Book


b = Book(123)
print(b)


Book(uid=123, price=0, weight=0, title='', author='')


Сделаем так, чтобы параметр current_id был уникальным для каждого нового объекта

In [3]:
@dataclass
class Goods:
    current_id = 0 # так как атрибут не аннотирован, он не попадает в инициализатор (видимо это что-то типа атрибута класса)

    uid: int = field(init=False)
    price: Any = None
    weight: Any = None

    def __post_init__(self):
        print('Goods: post init')
        Goods.current_id += 1
        self.uid = Goods.current_id

@dataclass
class Book(Goods):
    title: str = ''
    author: str = ''
    price: float = 0
    weight: int|float = 0

b1 = Book(1, 12, 'aaa', 'bbb')
print(b1)

b2 = Book(11, 1232, 'aaa', 'bbb')
print(b2)

Goods: post init
Book(uid=1, price=1, weight=12, title='aaa', author='bbb')
Goods: post init
Book(uid=2, price=11, weight=1232, title='aaa', author='bbb')


In [4]:
@dataclass
class Goods:
    current_id = 0 # так как атрибут не аннотирован, он не попадает в инициализатор (видимо это что-то типа атрибута класса)

    uid: int = field(init=False)
    price: Any = None
    weight: Any = None

    def __post_init__(self):
        print('Goods: post init')
        Goods.current_id += 1
        self.uid = Goods.current_id

@dataclass
class Book(Goods):
    title: str = ''
    author: str = ''
    price: float = 0
    weight: int|float = 0

    def __post_init__(self):
        print('Book: post init')

b1 = Book(1, 12, 'aaa', 'bbb')
print(b1)

b2 = Book(11, 1232, 'aaa', 'bbb')
print(b2)

Book: post init


AttributeError: 'Book' object has no attribute 'uid'

В данном случае был вызван метод __post_init__ дочернего класса, но не был вызван аналогичный метод в базовом классе, поэтому свойство не было сформировано. А не был он вызван потому что работает примерно также как init, то есть ищется в дочернем классе, и только если его там нет, он ищется в базовом

In [5]:
@dataclass
class Goods:
    current_id = 0 # так как атрибут не аннотирован, он не попадает в инициализатор (видимо это что-то типа атрибута класса)

    uid: int = field(init=False)
    price: Any = None
    weight: Any = None

    def __post_init__(self):
        print('Goods: post init')
        Goods.current_id += 1
        self.uid = Goods.current_id

@dataclass
class Book(Goods):
    title: str = ''
    author: str = ''
    price: float = 0
    weight: int|float = 0

    def __post_init__(self):
        super().__post_init__()
        print('Book: post init')

b1 = Book(1, 12, 'aaa', 'bbb')
print(b1)

Goods: post init
Book: post init
Book(uid=1, price=1, weight=12, title='aaa', author='bbb')


Еще усложним дочерний класс, добавив атрибут measure. Он по умолчанию принимает значение метода отдельного класса. Такой меод нельзя определить внутри класса, где он вызывается, поэтому пришлось создать отдельный вспомгателный класс

In [6]:
class GoodsMethodsFactory:
    @staticmethod
    def get_init_measure():
        return [0, 0, 0]



@dataclass
class Goods:
    current_id = 0 # так как атрибут не аннотирован, он не попадает в инициализатор (видимо это что-то типа атрибута класса)

    uid: int = field(init=False)
    price: Any = None
    weight: Any = None

    def __post_init__(self):
        print('Goods: post init')
        Goods.current_id += 1
        self.uid = Goods.current_id

@dataclass
class Book(Goods):
    title: str = ''
    author: str = ''
    price: float = 0
    weight: int|float = 0
    measure: list = field(default_factory=GoodsMethodsFactory.get_init_measure) # габарит предмета

    def __post_init__(self):
        super().__post_init__()
        print('Book: post init')

b1 = Book(1, 12, 'aaa', 'bbb')
print(b1)

Goods: post init
Book: post init
Book(uid=1, price=1, weight=12, title='aaa', author='bbb', measure=[0, 0, 0])


Функция make_dataclass. Допустим, мы хотим создать вот такой вот класс

In [13]:
from dataclasses import make_dataclass

class Car:
    def __init__(self, model, max_speed, price):
        self.model = model
        self.max_speed = max_speed
        self.price = price

    def get_max_speed(self):
        return self.max_speed

CarData = make_dataclass('CarData', [('model', str),
                                     'max_speed',
                                     ('price', float, field(default=0))],
                         namespace={'get_max_speed': lambda self: self.max_speed})

c = CarData('bmw', 256, 5555)
print(c)
print(c.get_max_speed())

CarData(model='bmw', max_speed=256, price=5555)
256


Такое объявление может понадобиться, когда требуется создать класс во время выполнения программы. В остальном проще объявлять класс по-нормальному